# SVM Regression (SVR) — Predicting `duration_minutes`
### Support Vector Regression with RBF, Linear & Polynomial Kernels

**Goal:** Build an SVR model to predict GitHub Actions workflow `duration_minutes` using YAML-derived and codebase features.

| Step | Description |
|------|-------------|
| 1 | Load & preprocess (type-cast, derive features) |
| 2 | IQR outlier removal on target |
| 3 | Encode categoricals (one-hot) |
| 4 | Feature selection (top features by EDA composite ranking) |
| 5 | Train/test split + StandardScaler |
| 6 | SVR with RBF, Linear, Poly kernels |
| 7 | GridSearchCV for hyperparameter optimisation |
| 8 | Evaluation: R², Adjusted R², MAE, RMSE, MAPE |
| 9 | Residual diagnostics |
| 10 | Hyperparameter tuning tips |

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
TARGET = 'duration_minutes'
print('Imports complete ✓')

---
## 1 · Load & Preprocess

In [ ]:
df_raw = pd.read_csv('comprehensive_features.csv')
print(f'Raw shape: {df_raw.shape}')
df_raw.head()

In [ ]:
# ── Drop excluded columns ──────────────────────────────────────────────
drop_cols = ['total_cost_usd', 'workflow_name', 'repo_name', 'head_sha']
df = df_raw.drop(columns=[c for c in drop_cols if c in df_raw.columns]).copy()
print(f'Shape after dropping identifiers/leaky cols: {df.shape}')

In [ ]:
# ── Type casting ──────────────────────────────────────────────────────
bool_cols = ['uses_matrix_strategy', 'is_using_setup_actions',
             'is_using_docker_actions', 'is_using_cache']
for c in bool_cols:
    if c in df.columns:
        df[c] = df[c].map({'True': 1, 'False': 0, True: 1, False: 0}).astype(int)

# fail_fast: coerce expression strings → True (default runtime value)
def parse_fail_fast(v):
    if v == 'True':  return 1
    if v == 'False': return 0
    return 1

if 'fail_fast' in df.columns:
    df['fail_fast'] = df['fail_fast'].apply(parse_fail_fast)

# container_image → binary flag
if 'container_image' in df.columns:
    df['has_container'] = (df['container_image'] != 'False').astype(int)
    df.drop(columns=['container_image'], inplace=True)

# yaml_depth to numeric
if 'yaml_depth' in df.columns:
    df['yaml_depth'] = pd.to_numeric(df['yaml_depth'], errors='coerce')

print('Type casting complete ✓')
df.dtypes

---
## 2 · IQR Outlier Removal on Target

In [ ]:
Q1, Q3 = df[TARGET].quantile([0.25, 0.75])
IQR = Q3 - Q1
lo, hi = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

n_before = len(df)
df = df[df[TARGET].between(lo, hi)].copy().reset_index(drop=True)
n_after = len(df)

print(f'Rows before: {n_before}')
print(f'Outliers removed: {n_before - n_after} (duration > {hi:.3f} min)')
print(f'Rows after: {n_after}')
print(f'Target range: {df[TARGET].min():.4f} – {df[TARGET].max():.4f} min')

---
## 3 · One-Hot Encode Categoricals

In [ ]:
cat_cols = ['os_label', 'primary_language']
cat_cols_present = [c for c in cat_cols if c in df.columns]

df = pd.get_dummies(df, columns=cat_cols_present, drop_first=True, dtype=int)
print(f'Shape after one-hot encoding: {df.shape}')

In [ ]:
# Handle any remaining NaN values
print(f'NaN counts before fill:\n{df.isnull().sum()[df.isnull().sum() > 0]}')
df = df.fillna(0)
print(f'\nFinal shape: {df.shape}')

---
## 4 · Feature Selection

SVR is computationally expensive (O(n²) to O(n³)). We use features identified as relevant by the EDA composite ranking. All features are used since one-hot categoricals are already sparse and informative.

In [ ]:
# Separate features and target
y = df[TARGET].values
X = df.drop(columns=[TARGET])

print(f'Feature count: {X.shape[1]}')
print(f'Features: {list(X.columns)}')

---
## 5 · Train/Test Split & Feature Scaling

**Feature scaling is critical for SVM** — the kernel function is distance-based, so unscaled features with large ranges dominate the model.

In [ ]:
# Use log1p-transformed target to stabilise variance (from EDA findings)
y_log = np.log1p(y)

X_train, X_test, y_train_log, y_test_log = train_test_split(
    X, y_log, test_size=0.2, random_state=RANDOM_STATE
)

# Keep original y_test for back-transformed metrics
y_test_original = np.expm1(y_test_log)
y_train_original = np.expm1(y_train_log)

# Standard scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train set: {X_train_scaled.shape[0]} samples, {X_train_scaled.shape[1]} features')
print(f'Test set:  {X_test_scaled.shape[0]} samples')

---
## 6 · SVR Training — Multiple Kernels

In [ ]:
def evaluate_model(model_name, y_true_log, y_pred_log, n_features):
    """Compute metrics in both log and original scale."""
    r2_log = r2_score(y_true_log, y_pred_log)
    n = len(y_true_log)
    adj_r2_log = 1 - (1 - r2_log) * (n - 1) / (n - n_features - 1)
    
    y_true_orig = np.expm1(y_true_log)
    y_pred_orig = np.expm1(y_pred_log)
    y_pred_orig = np.maximum(y_pred_orig, 0)
    
    r2_orig = r2_score(y_true_orig, y_pred_orig)
    mae = mean_absolute_error(y_true_orig, y_pred_orig)
    rmse = np.sqrt(mean_squared_error(y_true_orig, y_pred_orig))
    
    mask = y_true_orig > 0.01
    if mask.sum() > 0:
        mape = np.mean(np.abs((y_true_orig[mask] - y_pred_orig[mask]) / y_true_orig[mask])) * 100
    else:
        mape = np.nan
    
    metrics = {
        'Model': model_name,
        'R² (log)': round(r2_log, 4),
        'Adj R² (log)': round(adj_r2_log, 4),
        'R² (original)': round(r2_orig, 4),
        'MAE (min)': round(mae, 4),
        'RMSE (min)': round(rmse, 4),
        'MAPE (%)': round(mape, 2),
    }
    return metrics

print('Evaluation function defined ✓')

In [ ]:
# ── 6a. SVR with RBF Kernel (default) ────────────────────────────────
print('Training SVR (RBF kernel)...')
svr_rbf = SVR(kernel='rbf', C=1.0, epsilon=0.1, gamma='scale')
svr_rbf.fit(X_train_scaled, y_train_log)
y_pred_rbf = svr_rbf.predict(X_test_scaled)

metrics_rbf = evaluate_model('SVR-RBF', y_test_log, y_pred_rbf, X_train_scaled.shape[1])
print('SVR-RBF Results:')
for k, v in metrics_rbf.items():
    print(f'  {k}: {v}')

In [ ]:
# ── 6b. SVR with Linear Kernel ───────────────────────────────────────
print('Training SVR (Linear kernel)...')
svr_linear = SVR(kernel='linear', C=1.0, epsilon=0.1)
svr_linear.fit(X_train_scaled, y_train_log)
y_pred_linear = svr_linear.predict(X_test_scaled)

metrics_linear = evaluate_model('SVR-Linear', y_test_log, y_pred_linear, X_train_scaled.shape[1])
print('SVR-Linear Results:')
for k, v in metrics_linear.items():
    print(f'  {k}: {v}')

In [ ]:
# ── 6c. SVR with Polynomial Kernel ───────────────────────────────────
print('Training SVR (Poly kernel, degree=2)...')
svr_poly = SVR(kernel='poly', degree=2, C=1.0, epsilon=0.1, gamma='scale')
svr_poly.fit(X_train_scaled, y_train_log)
y_pred_poly = svr_poly.predict(X_test_scaled)

metrics_poly = evaluate_model('SVR-Poly(2)', y_test_log, y_pred_poly, X_train_scaled.shape[1])
print('SVR-Poly Results:')
for k, v in metrics_poly.items():
    print(f'  {k}: {v}')

---
## 7 · GridSearchCV — Optimise RBF SVR

In [ ]:
param_grid = {
    'C': [0.1, 1.0, 10.0, 100.0],
    'epsilon': [0.01, 0.05, 0.1, 0.2],
    'gamma': ['scale', 'auto', 0.01, 0.1],
}

print(f'Grid search: {np.prod([len(v) for v in param_grid.values()])} combinations × 3 folds')
print('This may take a few minutes...\n')

grid_search = GridSearchCV(
    SVR(kernel='rbf'),
    param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    verbose=1,
    return_train_score=True,
)
grid_search.fit(X_train_scaled, y_train_log)

print(f'\nBest parameters: {grid_search.best_params_}')
print(f'Best CV R² (log): {grid_search.best_score_:.4f}')

In [ ]:
# ── Evaluate best SVR ────────────────────────────────────────────────
best_svr = grid_search.best_estimator_
y_pred_best = best_svr.predict(X_test_scaled)

metrics_best = evaluate_model('SVR-RBF (tuned)', y_test_log, y_pred_best, X_train_scaled.shape[1])
print('Best SVR-RBF (tuned) Results:')
for k, v in metrics_best.items():
    print(f'  {k}: {v}')

In [ ]:
# ── Visualise GridSearchCV results (C vs epsilon heatmap) ─────────────
cv_results = pd.DataFrame(grid_search.cv_results_)

# Pivot for the best gamma
best_gamma = grid_search.best_params_['gamma']
subset = cv_results[cv_results['param_gamma'] == best_gamma]

if len(subset) > 0:
    pivot = subset.pivot_table(
        values='mean_test_score',
        index='param_C',
        columns='param_epsilon',
        aggfunc='mean'
    )
    
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlOrRd', ax=ax,
                linewidths=0.5)
    ax.set_title(f'GridSearchCV R² — RBF kernel (gamma={best_gamma})')
    ax.set_xlabel('Epsilon')
    ax.set_ylabel('C')
    plt.tight_layout()
    plt.show()

---
## 8 · Model Comparison & Cross-Validation

In [ ]:
# ── Summary table ────────────────────────────────────────────────────
results_df = pd.DataFrame([metrics_rbf, metrics_linear, metrics_poly, metrics_best])
print('=== SVR Model Comparison ===')
display(results_df)

In [ ]:
# ── Cross-validation on best model ───────────────────────────────────
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
X_all_scaled = scaler.fit_transform(X)

print('=== 5-Fold Cross-Validation R² (log1p target) ===')
svr_models = {
    'SVR-RBF (default)': SVR(kernel='rbf', C=1.0, epsilon=0.1, gamma='scale'),
    'SVR-Linear': SVR(kernel='linear', C=1.0, epsilon=0.1),
    'SVR-RBF (tuned)': SVR(**grid_search.best_params_, kernel='rbf'),
}

cv_scores = {}
for name, model in svr_models.items():
    scores = cross_val_score(model, X_all_scaled, y_log, cv=kf, scoring='r2')
    cv_scores[name] = scores
    print(f'  {name:25s}: mean R² = {scores.mean():.4f} ± {scores.std():.4f}  '
          f'[{scores.min():.4f}, {scores.max():.4f}]')

In [ ]:
# ── CV boxplot ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
ax.boxplot([cv_scores[m] for m in svr_models],
           labels=list(svr_models.keys()), patch_artist=True,
           boxprops=dict(facecolor='#58a6ff', alpha=0.7),
           medianprops=dict(color='white', lw=2))
ax.set_ylabel('R² Score (log1p target)')
ax.set_title('5-Fold CV Performance — SVR Variants')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 9 · Residual Diagnostics

In [ ]:
residuals_log = y_test_log - y_pred_best
y_pred_orig = np.expm1(y_pred_best)
y_pred_orig = np.maximum(y_pred_orig, 0)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Residual Diagnostics — Best SVR Model', fontsize=14)

# 1. Residuals vs Fitted (log scale)
ax = axes[0, 0]
ax.scatter(y_pred_best, residuals_log, alpha=0.3, s=12, color='#58a6ff')
ax.axhline(0, color='white', lw=1, ls='--')
ax.set_xlabel('Fitted values (log scale)')
ax.set_ylabel('Residuals (log scale)')
ax.set_title('Residuals vs Fitted')

# 2. Q-Q plot of residuals
ax = axes[0, 1]
(osm, osr), (slope, intercept, r_qq) = stats.probplot(residuals_log, dist='norm')
ax.scatter(osm, osr, color='#58a6ff', s=10, alpha=0.5)
xl = np.linspace(min(osm), max(osm), 100)
ax.plot(xl, slope * xl + intercept, color='#ff7b72', lw=2)
ax.set_xlabel('Theoretical Quantiles')
ax.set_ylabel('Sample Quantiles')
ax.set_title(f'Q-Q Plot (r={r_qq:.3f})')

# 3. Histogram of residuals
ax = axes[1, 0]
ax.hist(residuals_log, bins=50, color='#58a6ff', alpha=0.7, edgecolor='none', density=True)
x_range = np.linspace(residuals_log.min(), residuals_log.max(), 200)
ax.plot(x_range, stats.norm.pdf(x_range, residuals_log.mean(), residuals_log.std()),
        color='#ff7b72', lw=2, label='Normal fit')
ax.set_xlabel('Residuals (log scale)')
ax.set_ylabel('Density')
ax.set_title('Residual Distribution')
ax.legend()

# 4. Actual vs Predicted (original scale)
ax = axes[1, 1]
ax.scatter(y_test_original, y_pred_orig, alpha=0.3, s=12, color='#3fb950')
lim = max(y_test_original.max(), y_pred_orig.max())
ax.plot([0, lim], [0, lim], 'w--', lw=1.5, alpha=0.6, label='Perfect prediction')
ax.set_xlabel('Actual duration (min)')
ax.set_ylabel('Predicted duration (min)')
ax.set_title('Actual vs Predicted (original scale)')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Support vector analysis ──────────────────────────────────────────
n_sv = best_svr.n_support_ if hasattr(best_svr, 'n_support_') else None
total_sv = sum(best_svr.support_vectors_.shape) if hasattr(best_svr, 'support_vectors_') else None
n_train = X_train_scaled.shape[0]

print(f'Total training samples: {n_train}')
if hasattr(best_svr, 'support_vectors_'):
    n_svs = best_svr.support_vectors_.shape[0]
    print(f'Number of support vectors: {n_svs} ({n_svs/n_train*100:.1f}% of training data)')
    print(f'  → High % indicates the model found the problem complex')
    print(f'  → Consider increasing C to reduce support vectors (tighter fit)')

---
## 10 · Hyperparameter Tuning Tips for SVR

### Key Hyperparameters

#### `C` (Regularisation Parameter)
- Controls the trade-off between a smooth decision surface and classifying training points correctly.
- **Low C** → wider margin, more regularisation, underfitting risk.
- **High C** → narrower margin, less regularisation, overfitting risk.
- Typical range: `[0.01, 0.1, 1, 10, 100, 1000]`.
- Start with `C=1.0` and expand in log scale.

#### `epsilon` (ε-insensitive tube width)
- Predictions within ε of the true value incur **zero loss**.
- **Larger ε** → fewer support vectors, smoother model, larger errors tolerated.
- **Smaller ε** → more support vectors, tighter fit.
- Typical range: `[0.001, 0.01, 0.05, 0.1, 0.2, 0.5]`.
- Set proportional to expected noise level in the target.

#### `gamma` (RBF kernel coefficient)
- Defines how far the influence of a single training sample reaches.
- **Low gamma** → far reach, smooth decision boundary.
- **High gamma** → close reach, complex/wiggly boundary, overfitting risk.
- `'scale'` = `1 / (n_features × X.var())` — good default.
- `'auto'` = `1 / n_features` — often too high.
- Typical range: `[0.001, 0.01, 0.1, 1, 10]`.

#### `kernel`
- `'rbf'` — best default for non-linear relationships (use this first).
- `'linear'` — equivalent to Ridge regression; use when relationships are linear.
- `'poly'` — captures polynomial interactions; `degree=2` or `3`. Higher degrees are slow and overfit.

### General Tips
- **Always scale features** (StandardScaler or MinMaxScaler) — SVM is distance-based.
- **log1p transform** the target if right-skewed.
- SVR is **O(n² to n³)** in training — subsample if dataset > 10k rows.
- Use `GridSearchCV` or `RandomizedSearchCV` with `cv=3` for large grids.
- Monitor **support vector count** — if > 50% of training data, the model is struggling.
- For large datasets, consider `LinearSVR` (much faster than `SVR(kernel='linear')`).
- **Interaction features** can help linear SVR capture non-linear patterns without kernel tricks.

In [ ]:
print('═══ SVM REGRESSION SUMMARY ═════════════════════════════════════════════')
print()
best_row = results_df.loc[results_df['R² (log)'].idxmax()]
print(f'Best model: {best_row["Model"]}')
print(f'  R² (log scale):      {best_row["R² (log)"]}')
print(f'  Adj R² (log scale):  {best_row["Adj R² (log)"]}')
print(f'  R² (original scale): {best_row["R² (original)"]}')
print(f'  MAE:                 {best_row["MAE (min)"]} min')
print(f'  RMSE:                {best_row["RMSE (min)"]} min')
print(f'  MAPE:                {best_row["MAPE (%)"]}%')
print()
print(f'Best hyperparameters: {grid_search.best_params_}')
print(f'Features used: {X.shape[1]}')
print('\n✓ SVM Regression notebook complete.')